In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
#https://docs.langchain.com/oss/python/integrations/splitters/index


from unstructured.partition.auto import partition
from pathlib import Path

In [ ]:
path = Path(r"C:\Users\Admin\Desktop\ip\How To RAG\docs")

documents = []
id = 0

for docs in path.iterdir():
#print (docs)
    elements = partition (str(docs))
    #print("\n\n".join ([str(el) for el in elements]))  
    for el in elements:
        documents.append ({
                "id": f"{docs.name}_{id}",
                "text": el.text,
                "metadata": {
                    "source": docs.name,
                    "type": el.category,
                    "file_type": el.metadata.to_dict()["filetype"]
                    #"language": el.metadata.to_dict()["languages"]
            }
        })

        id += 1
#print (documents)

In [ ]:
display (documents)

#display (documents[1]["text"])
#print (documents[0]["metadata"])

#for item in documents:
    #print (item["metadata"]["source"])

In [ ]:
docs = [
    Document (
        page_content = item["text"], 
        metadata = {
            "id": item["id"],
            "source": item["metadata"]["source"],
            "type": item["metadata"]["type"],
            "file_type": item["metadata"]["file_type"]
        }
    )
    for item in documents
]

In [ ]:
display (docs)

In [ ]:
#https://reference.langchain.com/python/langchain-text-splitters

#RecursiveCharacterTextSplitter
#CharacterTextSplitter
#TokenTextSplitter
#SentenceTransformersTokenTextSplitter
#MarkdownHeaderTextSplitter
#HTMLHeaderTextSplitter
#PythonCodeTextSplitter
#RecursiveJsonSplitter

chunking = RecursiveCharacterTextSplitter (
    chunk_size = 800, #800 bom ponto de partida
    chunk_overlap = 100 #10 a 20% do chunk_size
) 

chunks = chunking.split_documents (docs)

In [ ]:
display (chunks)

<h3> Embedding </h3>

In [22]:
from langchain_huggingface import HuggingFaceEmbeddings #Para dar Load de um modelo de embeddings
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores

In [ ]:
embeddings = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Embedding Models") #https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
db = FAISS.from_documents(chunks, embeddings) #direto para vector database

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1602.85it/s]


In [ ]:
#display (db.docstore._dict)
#display (db.index.ntotal) #Número de Chunks
#print (db.index_to_docstore_id)
#db.save_local("x")

{0: '04c427e2-9e5d-41c8-b0af-99d1cb554c83', 1: 'a341945d-335c-411d-abde-1579d0d7a153', 2: 'ee9718bb-15db-4423-9b5b-8ceb7d803c60', 3: 'e23a001f-db76-4ab9-bda5-8673d4403892', 4: '4a7b9c00-40b8-4bf7-8716-3026439ca339', 5: 'c2d7f17f-0d28-4fbc-8f1e-7414564c973f', 6: 'c0b57e74-9f84-48da-b62d-fb265e069c6a', 7: '890a2d11-b046-475d-a3ed-aac2d539058a', 8: '4494ffe3-00eb-44ff-a65e-201cdc0f71f5', 9: '73b960d9-d368-479b-8770-e00342408487', 10: 'fabbe1e5-f59e-45de-9990-3004c6f71251', 11: 'b1e05d8f-2a65-4451-820a-936cd670ae9a', 12: 'f0d6b4ff-bba3-4498-ad34-7fb9d2d04fc2', 13: '1c0bc167-5ce0-480f-b49d-39fa6e5e94ad', 14: 'eca0c941-e313-4087-907e-03f75d126ff6', 15: 'b5593c6b-25c9-4bb9-b8e6-2c9f88c36e42', 16: 'ce677ce4-acd7-4e4a-94f7-ab65f5a29c82', 17: '75a7b5a3-cd73-42d4-8407-656f7705a9d5', 18: 'd1631a8c-32af-45e1-abc5-bf0b07ff52d0', 19: '250744e4-a1ad-47d6-9163-d729ba8a95de', 20: '22710db4-bb70-448d-96c4-21b34a953b7b', 21: '34c93a9d-2fb7-4b72-b71b-a5fdcba150fd', 22: '223afdcf-1683-43eb-b4c3-3451b8e1d85d

<h3> Embedding da Pergunta </h3>

In [ ]:
query = "Quem organiza a Copa do Mundo?"
embedding_vector = embeddings.embed_query(query)

print (embedding_vector)

[0.012974495999515057, 0.06915711611509323, -0.05095718428492546, 0.013340741395950317, -0.06273376941680908, 0.010590579360723495, 0.034257009625434875, 0.03308557718992233, 0.031079616397619247, 0.027060193940997124, 0.04907216131687164, -0.05698850378394127, 0.000848396448418498, 0.033951535820961, 0.014251581393182278, -0.06630183011293411, -0.008725435473024845, 0.08781328797340393, 0.036115098744630814, 0.02216370962560177, 0.019077621400356293, -0.047888435423374176, -0.046976488083601, 0.04763859510421753, -0.09870456159114838, 0.017720647156238556, 0.005008540116250515, 0.045122288167476654, 0.017329422757029533, -0.06113578379154205, -0.008207662962377071, 0.11331243813037872, 0.043781232088804245, 0.010197240859270096, 0.012585739605128765, 0.020364180207252502, 0.052182458341121674, -0.0789307951927185, 0.0007359012961387634, 0.04658599942922592, -0.02940898761153221, -0.0345708392560482, -0.008738677017390728, -0.07794294506311417, 0.03381083533167839, -0.04090262204408645

<h3> Similiariedade </h3>

In [43]:
busca = db.similarity_search ("Que país apoiou a invasão italiana da Etiópia quando outras grandes potências europeias não o fizeram?", k = 3)
busca2 = db.similarity_search_with_score ("Que país apoiou a invasão italiana da Etiópia quando outras grandes potências europeias não o fizeram?", k = 3)
busca3 = db.similarity_search_by_vector (embedding_vector, k = 3)


In [44]:
for i in busca2:
    print (i[1])
    print (i[0])

0.7446524
page_content='esperança de conter a Alemanha, o Reino Unido, a França e a Itália formaram a Frente de Stresa. A União Soviética, preocupada com os objetivos da Alemanha de ocupar vastas áreas do leste da Europa, escreveu um tratado de assistência mútua com a França. Antes de tomar efeito, porém, o pacto franco-soviético foi obrigado a passar pela burocracia da Liga das Nações, que o tornou essencialmente sem poder. No entanto, em junho de 1935, o Reino Unido fez um acordo naval independente com a Alemanha, flexibilizando as restrições anteriores. Os Estados Unidos, preocupados com os acontecimentos na Europa e na Ásia, aprovaram a Lei de Neutralidade em agosto. Em outubro, a Itália invadiu a Etiópia e a Alemanha foi o único grande país europeu que apoiou essa invasão. O governo italiano posteriormente' metadata={'id': 'Primeira Guerra Mundial.txt_1', 'source': 'Primeira Guerra Mundial.txt', 'type': 'NarrativeText', 'file_type': 'text/plain'}
0.8514353
page_content='Em outubro

In [ ]:
retriever = db.as_retriever(
    search_type = "similarity_score_threshold", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 5,
        "score_threshold": 0.50,
    }
)

retriever.get_relevant_documents("Em que ano Hitler repudiou o Tratado de Versalhes e acelerou o rearmamento alemão?")